<a href="https://colab.research.google.com/github/sevenjunebaby/Classification/blob/main/Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install praw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 9.4 MB/s eta 0:00:00


In [4]:
import praw
import pandas as pd

# ------------------------------
# 1️⃣ Configure Reddit API
# ------------------------------
reddit = praw.Reddit(
    client_id="tkaF9R88qm7k74WDz9wBxQ",
    client_secret="bFq8lTM-pCcmQVh_n0FRg1f2h-zw5g",
    user_agent="reddit_script"
)

# ------------------------------
# 2️⃣ Choose a subreddit
# ------------------------------
subreddit_name = "movies"   # change to any subreddit
subreddit = reddit.subreddit(subreddit_name)

# ------------------------------
# 3️⃣ Collect top posts
# ------------------------------
posts_data = []

for post in subreddit.hot(limit=50):  # change limit as needed
    posts_data.append({
        "title": post.title,
        "author": str(post.author),
        "upvotes": post.score,
        "num_comments": post.num_comments,
        "url": post.url
    })

# ------------------------------
# 4️⃣ Save to CSV
# ------------------------------
df = pd.DataFrame(posts_data)
df.to_csv(f"{subreddit_name}_posts.csv", index=False)
print(f"Saved {len(posts_data)} posts to {subreddit_name}_posts.csv")

# ------------------------------
# 5️⃣ Preview data
# ------------------------------
df.head()


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Saved 50 posts to movies_posts.csv


,title,author,upvotes,num_comments,url
0,"Hi /r/movies! I'm Samuel van Grinsven, writer/...",WentUpTheHillAMA,6,15,https://i.redd.it/md0ecudehisf1.png
1,"Hi reddit! I'm Mercedes Bryce Morgan, director...",BoneLakeAMA,493,235,https://i.redd.it/4jv2rfh6phsf1.png
2,"Jane Goodall, Iconic Chimpanzee Expert Who Was...",MarvelsGrantMan136,49943,692,https://variety.com/2025/film/news/jane-goodal...
3,Quentin Tarantino’s ‘Kill Bill: The Whole Bloo...,MarvelsGrantMan136,4924,431,https://www.thewrap.com/kill-bill-the-whole-bl...
4,Paul Thomas Anderson’s ‘One Battle After Anoth...,GoldDerby,1445,163,https://www.goldderby.com/film/2025/one-battle...


In [5]:
!pip install pandas scikit-learn


In [6]:

df['text'] = df['title'].astype(str)  # Use titles as text
df.head()


,title,author,upvotes,num_comments,url,text
0,"Hi /r/movies! I'm Samuel van Grinsven, writer/...",WentUpTheHillAMA,6,15,https://i.redd.it/md0ecudehisf1.png,"Hi /r/movies! I'm Samuel van Grinsven, writer/..."
1,"Hi reddit! I'm Mercedes Bryce Morgan, director...",BoneLakeAMA,493,235,https://i.redd.it/4jv2rfh6phsf1.png,"Hi reddit! I'm Mercedes Bryce Morgan, director..."
2,"Jane Goodall, Iconic Chimpanzee Expert Who Was...",MarvelsGrantMan136,49943,692,https://variety.com/2025/film/news/jane-goodal...,"Jane Goodall, Iconic Chimpanzee Expert Who Was..."
3,Quentin Tarantino’s ‘Kill Bill: The Whole Bloo...,MarvelsGrantMan136,4924,431,https://www.thewrap.com/kill-bill-the-whole-bl...,Quentin Tarantino’s ‘Kill Bill: The Whole Bloo...
4,Paul Thomas Anderson’s ‘One Battle After Anoth...,GoldDerby,1445,163,https://www.goldderby.com/film/2025/one-battle...,Paul Thomas Anderson’s ‘One Battle After Anoth...


In [7]:

df['label'] = ["Action","Drama","Comedy","Other","Horror","Action","Drama","Other","Comedy","Action"] + ["Other"]*(len(df)-10)


In [8]:
from sklearn.model_selection import train_test_split

X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english', max_features=500)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


In [10]:
from sklearn.naive_bayes import MultinomialNB

clf = MultinomialNB()
clf.fit(X_train_vec, y_train)


MultinomialNB()

In [11]:
y_pred = clf.predict(X_test_vec)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

       Other       1.00      1.00      1.00        10

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10



In [12]:
df_vec = vectorizer.transform(df['text'])
df['predicted_label'] = clf.predict(df_vec)

# Save to CSV
df.to_csv("movies_posts_classified_ml.csv", index=False)
print("Saved classified posts to movies_posts_classified_ml.csv")


Saved classified posts to movies_posts_classified_ml.csv


In [19]:
import pandas as pd

# Load your classified CSV
df = pd.read_csv("movies_posts_classified_ml.csv")

# Get unique classes in the dataset
class_counts = df['label'].value_counts()
print("All classes present in the data:")
print(class_counts)


All classes present in the data:
label
Other     42
Action     3
Drama      2
Comedy     2
Horror     1
Name: count, dtype: int64
